In [ ]:
import os
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tqdm
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

from custom.image_models import VitModel, example_config
from data.cifar_loader import CifarLoader

device = "cuda"

cifar_path = os.environ.get("CIFAR_PATH", "./data/cifar-100")
cifar_posttraining_dataset = CifarLoader(
    path = cifar_path,
    splits = "test",
    max_batch_size = 200,
    sample_rate = 0.02,
    num_example_per_label=1,
    shuffle_rate = 1.0,
)

num_examples = 5
random_images, random_labels = cifar_posttraining_dataset.random_example(num_examples)
fig, axes = plt.subplots(1, num_examples, figsize=(3 * num_examples, 3), dpi = 150)

# 遍历每个矩阵并显示
for i, ax in enumerate(axes):
    # 将 (3, 32, 32) 转换为 (32, 32, 3) 以便 matplotlib 显示
    img = np.transpose(random_images[i], (1, 2, 0))
    ax.imshow(img)
    ax.set_title(f"Image {i+1}")
    ax.axis('off')  # 关闭坐标轴

plt.tight_layout()
plt.show()

ckpt_path = "./checkpoints/cifar-100/vit.bin"
model_config = example_config
num_labels = 100
model_config.num_labels = num_labels
model = VitModel(model_config)
model.load_state_dict(torch.load(ckpt_path))


model.requires_grad_(False)
model.init_plugin(layer = -1, use_act = False)
model.plugin_model.enable_training()

In [ ]:
model.to(device)

cifar_posttraining_dataset.to(device)
num_epoch = 50
lr = 2e-2
scaler = torch.amp.GradScaler()
optimizer = optim.SGD(model.parameters(), lr = lr)
loss_hist = []
model.train()

for i in tqdm.tqdm(range(num_epoch)):
    # reset train dataloader
    cifar_posttraining_dataset._reset()
    # start training
    epoch_start_time = time.time()
    num_correct, num_train, num_loss = 0, 0, 0
    loss_total = 0.
    for batch in cifar_posttraining_dataset:
        input_feature, label = batch["input"], batch["label"]
        with torch.amp.autocast(device):
            output = model(input_feature, label)
        loss, label_predict = output.loss, output.label_predict
        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        num_train += label.size(0)
        num_loss += 1
        num_correct += torch.sum(label_predict == label).item()
        loss_total += loss.item()
        
    loss_hist.append(loss_total / num_loss)

plt.figure(figsize = (8, 4), dpi = 150)
plt.xlabel("iteration num", fontsize=14)
plt.ylabel("loss", fontsize=14)

print(len(loss_hist))
sns.lineplot(x = list(range(len(loss_hist))), y = loss_hist, zorder=1)

plt.show()

In [ ]:
model.to(device)
cifar_posttraining_dataset.to(device)
model.eval()
cifar_posttraining_dataset._reset()
module = model.plugin_model
X, Z, inputs, labels = [], [], [], []
for batch in cifar_posttraining_dataset:
    input_feature, label = batch["input"], batch["label"]
    output = model(input_feature, label)
    loss = output.loss
    X.append(module.cache['x'][:,-1,:])
    inputs.append(input_feature)
    labels.append(label)
    z = module.cache['z'].requires_grad_(True)
    grads = torch.autograd.grad(loss, [z])
    Z.append(grads[0][:,-1,:])

X = torch.cat(X, 0)
inputs = torch.cat(inputs, 0)
labels = torch.cat(labels, 0)
Z = torch.cat(Z, 0)

print(X.size(), Z.size(), inputs.size(), labels.size())

In [ ]:
class RewritingModel(object):
    def __init__(self, model, lambda_1 = 1., lambda_2 = 1.) -> None:
        super().__init__()
        self.lambda_1, self.lambda_2 = lambda_1, lambda_2
        module = model.plugin_model
        self.K0 = module.cache['K0'].to(device).T
        self.V0 = module.cache['V0'].to(device).T
        self.K = module.fc_in.weight.T - self.K0
        self.V = module.fc_out.weight.T - self.V0

    def solve_MLE(self, A, B):
        # min ||AX - B||^2
        return A.T @ torch.linalg.pinv(A @ A.T) @ B
    
    def solve_Procrustes(self, A, B, C, D):
        device = A.device
        U, _, V_T = torch.linalg.svd(self.lambda_1 * B.T @ A + self.lambda_2 * D.T @ C)
        I_uv = torch.eye(V_T.size(0), U.size(0), device = device)
        Omega = V_T.T @ I_uv @ U.T
        return Omega
    
    def compute_coefficient(self, X, Y, C):
        K = self.K
        V = self.V
        # (n,d) * (d.m) * (m,n)
        Sigma_K = torch.diag(X @ K @ C) / torch.diag(X @ X.T)
        # (n,m) * (m,d) * (d,n)
        Sigma_V = torch.diag(C.T @ V @ Y.T) / torch.diag(Y @ Y.T)
        return Sigma_K, Sigma_V
    
    def rewrite_KV(self, X, Y):
        K = self.K
        V = self.V
        C = self.solve_Procrustes(K, X.T, V.T, Y.T)
        Sigma_K, Sigma_V = self.compute_coefficient(X, Y, C)
        return C, Sigma_K, Sigma_V

# l1, l2 = 1 / X.norm(-1).mean(), 1 / Z.norm(-1).mean()
l1, l2 = 1, 0
RM = RewritingModel(model, l1, l2)
# C, Sigma_K, Sigma_V = RM.rewrite_KV(X, Z)
C_K = RM.solve_MLE(X.T, RM.K)
Sigma_K, Sigma_V = torch.ones((X.size(0),), device = device), torch.ones((X.size(0),), device = device)
C_V = RM.solve_MLE(X.T, RM.K0 @ RM.V)
eta = torch.diag(RM.K0 @ RM.K0.T).mean()

print(C_K.size(), C_V.size(), Sigma_K.size(), Sigma_V.size())
print(
    torch.norm(RM.K - X.T @ torch.diag(Sigma_K) @ C_K).item(), 
    torch.norm(RM.V - (RM.K0.T / eta) @ X.T @ C_V).item(),
)

In [ ]:
@torch.no_grad()
def ppl_delta_with_C(model, inputs, y, idx, cache, kv = 'KV'):
    input_i = inputs[idx:idx+1]
    yi = y[idx:idx+1]
    inputs = torch.cat([inputs[:idx], inputs[idx+1:]], 0)
    y = torch.cat([y[:idx], y[idx+1:]], 0)

    output_i = model(input_i, yi)
    output = model(inputs, y)

    logits_old_i = output_i.logits
    loss_old_i = output_i.loss
    logits_old = output.logits
    loss_old = output.loss

    C_K, C_V = cache['C_K'], cache['C_V']
    ck_i, cv_i = C_K[idx:idx+1], C_V[idx:idx+1]
    # I_m = torch.eye(c_i.size(0), device = c_i.device)
    # M = (I_m - c_i @ c_i.T)
    x, sigma_k, sigma_v = cache['X'], cache['Sigma_K'], cache['Sigma_V']
    K0, eta = cache['K0'], cache['eta']
    dK = sigma_k[idx] * (ck_i.T @ x[idx].unsqueeze(0))
    dV = sigma_v[idx] * ((K0.T / eta) @ x[idx].unsqueeze(1) @ cv_i).T
    
    KV_cache = [
        model.plugin_model.fc_in.weight.clone(),
        model.plugin_model.fc_out.weight.clone(),
    ]
    
    # print(torch.argmax(x[idx] @ KV_cache[0].T @ C).item())

    if 'K' in kv:
        nK = KV_cache[0] - dK
    else:
        nK = KV_cache[0]
    if 'V' in kv:
        nV = KV_cache[1] - dV
    else:
        nV = KV_cache[1]
        
    model.plugin_model.fc_in.weight.copy_(nK)
    model.plugin_model.fc_out.weight.copy_(nV)
    
    output_i = model(input_i, yi)
    output = model(inputs, y)

    logits_new_i = output_i.logits
    loss_new_i = output_i.loss
    logits_new = output.logits
    loss_new = output.loss

    model.plugin_model.fc_in.weight.copy_(KV_cache[0])
    model.plugin_model.fc_out.weight.copy_(KV_cache[1])
    return (
        loss_old_i.item(), 
        loss_new_i.item(),
        loss_old.item(), 
        loss_new.item(),
    ), (
        logits_old_i,
        logits_new_i,
        logits_old,
        logits_new,
    )

cache = {
    "C_K": C_K,
    "C_V": C_V,
    "Sigma_K": Sigma_K,
    "Sigma_V": Sigma_V,
    "X": X,
    "K0": RM.K0,
    "V0": RM.V0,
    "eta": eta,
}
case_results = []
for i in range(X.size(0)):
    outputs, output_logits = ppl_delta_with_C(model, inputs, labels, i, cache, 'V')
    case_results.append({
        'old': output_logits[0].squeeze().cpu(), 
        'new': output_logits[1].squeeze().cpu(), 
        'ans': labels[i],
        'old_local': output_logits[2], 
        'new_local': output_logits[3],
        'local_ans': torch.cat([labels[:i], labels[i+1:]], 0),
    })
    print("To erase (before -> after): ({:.6f} -> {:.6f}), To retain (before -> after): ({:.6f} -> {:.6f})".format(*outputs))

In [ ]:
from src.evaluate import (
    erase_acc, 
    summarize_acc, 
    ppl, 
    locality, 
)

def eval_case_results(case_results):
    case_metrics = []
    for case_result in case_results:
        case_metric = dict()
        case_metric.update(erase_acc(case_result))
        case_metric.update(ppl(case_result))
        case_metric.update(locality(case_result))

        case_metrics.append(case_metric)

    result_dict = dict()
    for key in case_metrics[0].keys():
        if key in ("T2T", "T2F", "F2T", "F2F"):
            result_dict[key] = np.sum([line[key] for line in case_metrics])
        else:
            result_dict[key] = np.mean([line[key] for line in case_metrics])
    result_dict = summarize_acc(result_dict)

    print(
        '\n'.join(
            ["{}: {:.4f}".format(key, value) 
            for (key, value) in result_dict.items()]
        )
    )
    return result_dict

res = eval_case_results(case_results)

In [ ]:
def locate_kn_ig(
    model,
    source_target,
):
    def scaled_input(emb, num_cuts):
        # emb: (1, ffn_size)
        baseline = torch.zeros_like(emb)  # (1, ffn_size)
        step = (emb - baseline) / num_cuts  # (1, ffn_size)
        res = torch.cat([torch.add(baseline, step * i) for i in range(num_cuts)], dim=0)  # (num_points, ffn_size)
        return res, step[0]
    
    num_cuts = 20
    ig_grads = []
    for dic in tqdm.tqdm(source_target):
        x, y = dic['x'], dic['y']
        nx_dim = len(x.size())
        repeated_x = x.unsqueeze(0).repeat([num_cuts] + [1] * nx_dim)

        output = model.forward(x.unsqueeze(0), return_inter=True)
        inter = output.intermediate

        scaled_weights, weights_step = scaled_input(inter, num_cuts)
        scaled_weights.requires_grad_(True)
        
        z = model(repeated_x, custom_inter = scaled_weights)
        logits = z.logits
        tgt_prob = F.softmax(logits, -1)
        gradient = torch.autograd.grad(torch.unbind(tgt_prob[:, y]), scaled_weights)[0]
        ig_grad = gradient.sum(dim = 0) * weights_step
        ig_grads.append(ig_grad)
    ig_grads = torch.stack(ig_grads)

    return ig_grads

xy = [{'x':inputs[i], 'y':labels[i]} for i in range(labels.size(0))]
ig = locate_kn_ig(model, xy)
print(ig.size())

In [ ]:
@torch.no_grad()
def ppl_delta_with_kn(model, x, s, ig, idx):
    xi = x[idx:idx+1]
    si = s[idx:idx+1]
    x = torch.cat([x[:idx], x[idx+1:]], 0)
    s = torch.cat([s[:idx], s[idx+1:]], 0)

    zi = model.forward(xi, si)
    logits_old_i = zi.logits
    loss_old_i = zi.loss

    z = model.forward(x, s)
    logits_old = z.logits
    loss_old = z.loss

    # I = xi @ model.K @ C + model.bias_K @ C
    I = ig[idx]

    kn = torch.argmax(I, -1)
    # kn = idx

    KV_cache = [
        model.plugin.fc_in.weight.clone(),
        model.plugin.fc_out.weight.clone(),
    ]
    
    # print(torch.argmax(x[idx] @ KV_cache[0].T @ C).item())

    nK = KV_cache[0].clone()
    nK[kn, :] *= 0.
    nV = KV_cache[1].clone()
    nV[:, kn] *= 0.
        
    model.plugin.fc_in.weight.copy_(nK)
    model.plugin.fc_out.weight.copy_(nV)
  
    zi = model(xi, si)
    logits_new_i = zi.logits
    loss_new_i = zi.loss

    z = model(x, s)
    logits_new = z.logits
    loss_new = z.loss

    model.plugin.fc_in.weight.copy_(KV_cache[0])
    model.plugin.fc_out.weight.copy_(KV_cache[1])

    return (
        loss_old_i.item(), 
        loss_new_i.item(),
        loss_old.item(), 
        loss_new.item(),
    ), (
        logits_old_i,
        logits_new_i,
        logits_old,
        logits_new,
    )

case_results = []
for i in range(inputs.size(0)):
    outputs, output_logits = ppl_delta_with_kn(model, inputs, labels, ig, i)
    case_results.append({
        'old': output_logits[0].squeeze().cpu(), 
        'new': output_logits[1].squeeze().cpu(), 
        'ans': labels[i],
        'old_local': output_logits[2], 
        'new_local': output_logits[3],
        'local_ans': torch.cat([labels[:i], labels[i+1:]], 0),
    })
    print("To erase (before -> after): ({:.6f} -> {:.6f}), To retain (before -> after): ({:.6f} -> {:.6f})".format(*outputs))

In [ ]:
from src.evaluate import (
    erase_acc, 
    summarize_acc, 
    ppl, 
    locality, 
)

def eval_case_results(case_results):
    case_metrics = []
    for case_result in case_results:
        case_metric = dict()
        case_metric.update(erase_acc(case_result))
        case_metric.update(ppl(case_result))
        case_metric.update(locality(case_result))

        case_metrics.append(case_metric)

    result_dict = dict()
    for key in case_metrics[0].keys():
        if key in ("T2T", "T2F", "F2T", "F2F"):
            result_dict[key] = np.sum([line[key] for line in case_metrics])
        else:
            result_dict[key] = np.mean([line[key] for line in case_metrics])
    result_dict = summarize_acc(result_dict)

    print(
        '\n'.join(
            ["{}: {:.4f}".format(key, value) 
            for (key, value) in result_dict.items()]
        )
    )
    return result_dict

res = eval_case_results(case_results)

In [ ]:
import json, math, random

@torch.no_grad()
def ppl_delta_with_shuffle(model, inputs, y, permutation, cache, KV = 'K', return_logits = False):
    y_shuffle = y[permutation]
    output_shuffle_old = model(inputs, y_shuffle, True)
    output_unshuffle_old = model(inputs, y, True)

    xy_shuffle_loss_old = output_shuffle_old.loss
    xy_unshuffle_loss_old = output_unshuffle_old.loss

    C_K, C_V = cache['C_K'], cache['C_V']
    # I_m = torch.eye(c_i.size(0), device = c_i.device)
    # M = (I_m - c_i @ c_i.T)
    x = cache['X']
    # x_shuffle = x[permutation]
    K0, V0, eta = cache['K0'], cache['V0'], cache['eta']
    dK, dV = (x.T @ C_K), (K0.T / eta) @ x.T @ C_V
    if 'K' in KV:
        dK = (x.T @ C_K[permutation])
    if 'V' in KV:
        dV = (K0.T / eta) @ x.T @ C_V[permutation]
    
    KV_cache = [
        model.plugin.fc_in.weight.clone(),
        model.plugin.fc_out.weight.clone(),
    ]
    
    # print(torch.argmax(x[idx] @ KV_cache[0].T @ C).item())

    nK = K0 + dK
    nV = V0 + dV
        
    model.plugin.fc_in.weight.copy_(nK.T)
    model.plugin.fc_out.weight.copy_(nV.T)

    output_shuffle_new = model(inputs, y_shuffle, True)
    output_unshuffle_new = model(inputs, y, True)

    xy_shuffle_loss_new = output_shuffle_new.loss
    xy_unshuffle_loss_new = output_unshuffle_new.loss

    model.plugin.fc_in.weight.copy_(KV_cache[0])
    model.plugin.fc_out.weight.copy_(KV_cache[1])

    outputs = (
        xy_shuffle_loss_old, 
        xy_unshuffle_loss_old,
        xy_shuffle_loss_new, 
        xy_unshuffle_loss_new,
    )
    if return_logits:
        outputs += (output_unshuffle_old.logits, output_unshuffle_new.logits)
    return outputs

cache = {
    "C_K": C_K,
    "C_V": C_V,
    "X": X,
    "K0": RM.K0,
    "V0": RM.V0,
    "eta": eta,
}

result = []
nums = [2,3,5,8,13,21,34,55,89]
for num_shuffled in nums:
    if num_shuffled == 0:
        result.append([0,0,0])
        continue
    split_idx = num_shuffled // 2
    permutation = list(range(split_idx,num_shuffled)) + list(range(0, split_idx)) + list(range(num_shuffled, X.size(0)))
    permutation = torch.tensor(permutation, device=device)
    permuted_idx = torch.arange(0, num_shuffled, device=device)
    sorted_idx = torch.arange(num_shuffled, X.size(0), device=device)
    LSOA, LUOA, LSNA, LUNA = ppl_delta_with_shuffle(model, inputs, labels, permutation, cache, 'K')

    LSO, LUO, LSN, LUN = (x[permuted_idx] for x in (LSOA, LUOA, LSNA, LUNA))
    shuffled_target_ld = torch.mean(LSN - LSO).item()
    shuffled_target_ac = torch.mean((LSN < LSO).float()).item()
    original_target_ld = torch.mean(LUN - LUO).item()
    original_target_ac = torch.mean((LUN > LUO).float()).item()

    LSO, LUO, LSN, LUN = (x[sorted_idx] for x in (LSOA, LUOA, LSNA, LUNA))
    locality_target_ld = torch.mean(abs(LUN - LUO)).item()
    result.append([shuffled_target_ld, original_target_ld, locality_target_ld])

with open("./outputs/shuffle.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

num_shuffle = 12

def get_derangement(n):
    while True:
        p = torch.randperm(n)
        if (p == torch.arange(n)).any() == False:
            return p

permutation = torch.cat((get_derangement(num_shuffle).to(device), torch.arange(num_shuffle, X.size(0), device=device)))
perm_map = torch.eye(num_shuffle, device=device)[permutation[:num_shuffle]]
logits_map_old, logits_map_new = ppl_delta_with_shuffle(model, inputs, labels, permutation, cache, 'V', True)[-2:]
loss_map_old, loss_map_new = map(lambda x:x[:num_shuffle,:num_shuffle].cpu().tolist(), [logits_map_old, logits_map_new])

with open("./outputs/shuffle_map.json", "w", encoding="utf-8") as f:
    json.dump({"old": loss_map_old, "new": loss_map_new, "ref": perm_map.cpu().tolist()}, f, ensure_ascii=False, indent=4)

